<p> <center> <a href="../../Start-NIM-RAG.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="rag_nim_endpoints.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="rag_nim_endpoints.ipynb">1</a>
        <a >2</a>
        <a href="nim_lora_adapter.ipynb">3</a>
        <!-- <a href="challenge.ipynb">4</a> -->
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="nim_lora_adapter.ipynb">Next Notebook</a></span>
</div>

# ローカライズされたNVIDIA NIMによるRAGの構築
---


このNotebookでは、ローカライズされたNVIDIA NIM(NVIDIA Inference Microservice)を使用してRAG (検索拡張生成)パイプラインを構築するデモンストレーションを行います。まず初めに、NVIDIA API Keyの設定、NIMイメージの取得とデプロイ、ローカルにデプロイされたNIMを使用するRAGアプリケーションの構築について説明します。

### NVIDIA API キーの設定

前回のNotebookでは、生成したNVIDIA API KEYの設定方法を学びました。このNotebookの要件として、NVIDIA NIM の docker イメージを取得する為に、環境変数 `NVIDIA_API_KEY` としてキーを設定する必要があります。まだキーを取得していない場合は、NVIDIA NIM API [ホームページ](https://build.nvidia.com/explore/discover) にアクセスしてAPIキーを生成してください。以下のセルを実行し、表示されるテキストボックスにNVIDIA API KEYを入力し、キーボードのEnterキーを押してください。

In [ ]:
import os
import getpass

if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    nvapi_key = getpass.getpass("Enter your NVIDIA API key: ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
    os.environ["NGC_API_KEY"] = nvapi_key


### セルフホスト型NIM

以下のセルを実行して、dockerデーモンが起動していることを確認してください。

In [ ]:
! docker ps

**期待される出力 (実行中のコンテナが存在しない場合):**

```python

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES

```

### NVCR (NVIDIA Container Registry)へのログイン

NVIDIA NIM の docker イメージにアクセスするには、`docker login nvcr.io`でログインする必要があります。このプロセスでは、`--username $oauthtoken`としてのデフォルトのユーザ名と、`$NGC_API_KEY`の値を受け付ける `--password-stdin` が必要です。

In [ ]:
! echo -e "$NGC_API_KEY" | docker login nvcr.io --username '$oauthtoken' --password-stdin

**想定される出力**:
```
WARNING! Your password will be stored unencrypted in /home/yagupta/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store

Login Succeeded
```

### NVIDIA NIMの選択

[NIMs Catalog](https://build.nvidia.com/explore/reasoning)には、さまざまなドメインにおける複数の最先端モデルが掲載されています。下のスクリーンショットのように、`RUN ANYWHERE`タグの付いたものを探してください。これらのNIMイメージはダウンロード可能で、モデルや必要な最適化ランタイムを含んでおり、すぐに使い始めるのに役立ちます。

<img src="imgs/catalog1.jpg" style="width: 900px; height: auto;">

お好みのNIMモデルを選択し、dockerタブをクリックし、以下のスクリーンショットのように赤枠内のイメージ名をコピーします。 

<img src="imgs/catalog2.jpg" style="width: 900px; height: auto;">

### ImageのPull

次のステップはDockerイメージをPullすることです。ここでは、`llama3-3.1-swallow-8b-instruct-v0.1`をプルすることでこのステップを実演します。

In [ ]:
! docker pull nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1:latest


**想定される出力:** (すでにImageを取得している場合)
```python

latest: Pulling from nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1
Digest: sha256:00dbe414f31d19cb63b09795d40e74bb309456b9cb32f580110a20e4473849d6
Status: Image is up to date for nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1:latest
nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1:latest
```

利用可能なイメージをリストアップして、モデルイメージをチェックしてみましょう。*`IMAGE ID`は、以下の期待される出力の下に表示されるものと異なる可能性があることに注意してください*。

In [ ]:
! docker image ls

**想定される出力**:

```python
REPOSITORY         TAG       IMAGE ID       CREATED        SIZE
nvcr.io/nim/meta/llama-3.1-swallow-8b-instruct-v0.1     latest     3882ef11dbcc   5 months ago   18.9GB
```


#### モデル アーティファクトの為のキャッシュの設定

NVIDIA NIMは、ハードウェア上で最大のパフォーマンスを達成するために最適なプロファイルが選択されるように、多くのファイルをダウンロードします。モデル成果物をキャッシュする場所を `LOCAL_NIM_CACHE` として設定し、その変数をエクスポートします。

In [ ]:
from os.path import expanduser
home = expanduser("~")
os.environ['LOCAL_NIM_CACHE']=f"{home}/.cache/nim"
!echo $LOCAL_NIM_CACHE

In [ ]:
!mkdir -p "$LOCAL_NIM_CACHE"
# !chmod 777 "$LOCAL_NIM_CACHE"

### NIM LLM マイクロサービスの起動

以下のセルを実行し docker run コマンドを実行して NIM LLM マイクロサービスを起動します。

```python
docker run -it --rm -d --gpus all --name=llm_nim --shm-size=16GB  -e NGC_API_KEY  -v '$LOCAL_NIM_CACHE':/opt/nim/.cache  -u $(id -u) -p 8000:8000  nvcr.io/nim/meta/llama-3.1-swallow-8b-instruct-v0.1
```

このDockerコマンドは、以下のフラグを使ってNIM LLMマイクロサービスを起動します:

- `-it`: 擬似TTYを割り当て、対話プロセス用にSTDINをオープンしておきます
- `--rm`:  終了時にコンテナを自動的に削除します
- `-d`: コンテナをデタッチモード（バックグラウンド）で実行します
- `--gpus all`: コンテナをデタッチモード（バックグラウンド）で実行する： コンテナが利用可能なすべての GPU にアクセスできるようにします
- `--name=llm_nim`: コンテナに"llm_nim"という名前をつけます
- `--shm-size=16GB`: `/dev/shmの`サイズを 16GB に設定します
- `-e NGC_API_KEY`: 環境変数 NGC_API_KEY をコンテナに渡します
- `-v $LOCAL_NIM_CACHE:/opt/nim/.cache`: ローカル NIM キャッシュディレクトリをコンテナの /opt/nim/.cache にマウントします
- `-u $(id -u)`: 現在のユーザーの UID でコンテナを実行します
- `-p 8000:8000`: ホストのポート 8000 をコンテナのポート 8000 にマップします
- `nvcr.io/nim/meta/llama-3.1-swallow-8b-instruct-v0.1`: 使用するDockerイメージを指定します

システムは複数のプロセスを実行することができるので、実行中のアプリケーションでポートを占有しないようにしなければなりません。以下のコードでは、一意の空きポートを見つけて割り当てています：

In [ ]:
import random
import socket

def find_available_port(start=11000, end=11999):
    while True:
        # Randomly select a port between start and end range
        port = random.randint(start, end)
        
        # Try to create a socket and bind to the port
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            try:
                sock.bind(("localhost", port))
                # If binding is successful, the port is free
                return port
            except OSError:
                # If binding fails, the port is in use, continue to the next iteration
                continue

# Find and print an available port
os.environ['CONTAINER_PORT'] = str(find_available_port())
print(f"Your have been alloted the available port: {os.environ['CONTAINER_PORT']}")

In [ ]:
! docker run -it -d --rm \
--gpus all \
--name=llm_nim \
--shm-size=16GB  \
-e $NGC_API_KEY \
-v $LOCAL_NIM_CACHE:/opt/nim/.cache \
-u $(id -u) \
-p $CONTAINER_PORT:8000 \
nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1:latest

# In order to ensure, the local NIM container is completely loaded and doesn't remain in pending stage, we instantiate a wait interval
! sleep 60

In [ ]:
! docker logs --tail 45 llm_nim

**想定される出力:**
```
0.0.0.0:8000/v1/metrics
  0.0.0.0:8000/v1/health/ready
  0.0.0.0:8000/v1/health/live
  0.0.0.0:8000/v1/models
  0.0.0.0:8000/v1/license
  0.0.0.0:8000/v1/metadata
  0.0.0.0:8000/v1/manifest
  0.0.0.0:8000/v1/version
  0.0.0.0:8000/v1/chat/completions
  0.0.0.0:8000/v1/completions
  0.0.0.0:8000/experimental/ls/inference/chat_completion
  0.0.0.0:8000/experimental/ls/inference/completion
INFO 2025-05-26 07:20:10.123 api_server.py:663] An example cURL request:
curl -X 'POST' \
  'http://0.0.0.0:8000/v1/chat/completions' \
  -H 'accept: application/json' \
  -H 'Content-Type: application/json' \
  -d '{
    "model": "tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1",
    "messages": [
      {
        "role":"user",
        "content":"Hello! How are you?"
      },
      {
...
INFO 2025-05-26 07:20:10.151 server.py:82] Started server process [66]
INFO 2025-05-26 07:20:10.152 on.py:48] Waiting for application startup.
INFO 2025-05-26 07:20:10.154 on.py:62] Application startup complete.
INFO 2025-05-26 07:20:10.155 server.py:214] Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)

```

### クイックテストの開始
NIMが稼働していることを2つの方法で素早くテストできます:
- LangChain NVIDIA Endpoints
- 単純なOpenAI完了リクエスト

**パラメータの説明:**
- **base_url**:  NVIDIA NIM の docker イメージがデプロイされているURL
- **model**: デプロイされた NVIDIA NIM モデルの名前
- **temperature**: サンプリングのランダム性を調整する。温度を下げると、高い確率で単語が選択される可能性が高くなります。
- **top_p**: モデルがどの程度決定論的かを制御します。正確で事実に基づいた回答を求める場合は、この値を低く保ちます。より多様な回答を求める場合は、値を大きくします
- **max_tokens**: 生成される出力トークンの最大数


In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(base_url="http://0.0.0.0:{}/v1".format(os.environ['CONTAINER_PORT']), model="tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1", temperature=0.1, max_tokens=1000, top_p=1.0)

result = llm.invoke("フランスの首都はどこですか？")
print(result.content)

エラーが出力された場合は、しばらく待ってから上記のセルを再実行してください。エラーは、NIMコンテナが完全に立ち上がっていないことが原因かもしれません。

In [ ]:
!curl -X 'POST' \
    "http://0.0.0.0:${CONTAINER_PORT}/v1/completions" \
    -H "accept: application/json" \
    -H "Content-Type: application/json" \
    -d '{"model": "tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1", "prompt": "フランスの首都はどこですか?", "temperature": 0.1, "top_p": 1.0, "max_tokens": 64}'

### RAG アプリケーション

このセクションでは、前のノートブックの手順に従って、ローカルに配置されたNVIDIA NIMをベースとしたRAGアプリケーションを構築します。このデモでは、前のノートブックのように2つのLLMを使って会話型検索チェーンを作るのではなく、1つのLLM `llama-3.1-swallow--8b-instruct` を使って会話型検索チェーンを作ります。これは各NIMイメージには1つのベースモデルがあるからです。ローカルに配置されたNIMとリモートアクセスを使用することも可能ですが、わかりやすくするために、単一のLLMのアプローチにこだわります。 

#### ライブラリのインポート

In [ ]:
from langchain.chains import ConversationalRetrievalChain, LLMChain
from langchain.chains.conversational_retrieval.prompts import CONDENSE_QUESTION_PROMPT, QA_PROMPT
from langchain.chains.question_answering import load_qa_chain
from langchain.memory import ConversationBufferMemory
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

#### ウェブリンク・データソースの作成

お好みのウェブリンクを差し替えたり、追加したりすることができます。

In [ ]:
urls = [
"https://www.fsa.go.jp/news/r5/20230829/230829_main.pdf",
"https://www.tioj.or.jp/activity/pdf/190619_06.pdf",
"https://www.pmda.go.jp/files/000251332.pdf",
"https://www.jili.or.jp/files/research/zenkokujittai/pdf/r3/i-xvii.pdf",
"https://www.zenginkyo.or.jp/fileadmin/res/news/news350331_1.pdf"
]


## 実装

In [ ]:
!uv pip install pypdf

#### PDF ファイルを読み込む関数を作る

以下は、PDFファイルを読み込むためのヘルパー関数です。

In [ ]:
from pypdf import PdfReader
from io import BytesIO
import requests
import re

def clean_japanese_pdf_text(text: str) -> str:
    # 日本語文中の改行を削除
    text = re.sub(r'(?<=[ぁ-んァ-ヶ一-龥])\s*\n\s*(?=[ぁ-んァ-ヶ一-龥])', '', text)
    # 連続するスペースや全角スペースを1つにまとめる
    text = re.sub(r'[ 　]{2,}', ' ', text)
    # 各行の先頭空白を削除
    text = re.sub(r'^\s+', '', text, flags=re.MULTILINE)
    # 「..... 4」形式の目次行のページ番号を除去
    text = re.sub(r'\.{3,}\s*\d{1,3}', '', text)
    text = text.replace("\n", "")
    return text

def pdf_document_loader(url: str) -> str:
    """
    Loads and extracts cleaned text from a PDF at the given URL using `pypdf`.

    Args:
        url: The URL of the PDF.

    Returns:
        Cleaned text content of the PDF.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
        reader = PdfReader(BytesIO(response.content))
        text = ""
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted
        return clean_japanese_pdf_text(text.strip())
    except Exception as e:
        print(f"Failed to load {url} due to: {e}")
        return ""



#### 埋め込みとDocument Text Splitterの作成

埋め込みを保存するパスを初期化し、`pdf_document_loader`関数を実行し、ドキュメントをテキストの塊に分割する関数を作ってみましょう。

In [ ]:
from tqdm import tqdm

def create_embeddings(embeddings_model,embedding_path: str = "./embed"):

    embedding_path = "./embed"
    print(f"Storing embeddings to {embedding_path}")

    documents = []
    for url in tqdm(urls):
        print(url)
        document = pdf_document_loader(url)
        #document = html_document_loader(url)
        documents.append(document)


    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=0,
        length_function=len,
    )
    print("Total documents:",len(documents))
    texts = text_splitter.create_documents(documents)
    print("Total texts:",len(texts))
    index_docs(embeddings_model,url, text_splitter, texts, embedding_path,)
    print("Generated embedding successfully")

#### LangChainからNVIDIA AI Endpointsを使って埋め込みを生成する

このセクションでは、LangChain用のNVIDIA AI Endpointsを使って埋め込みを生成し、将来の再利用のためにエンベッディングを`/embed`ディレクトリのオフラインベクタストアに保存する方法をデモします。

In [ ]:
embeddings_model = NVIDIAEmbeddings(model="NV-Embed-QA") # or use nvidia/nv-embedqa-e5-v5

以下では、ドキュメントページのコンテンツをループしてテキストとメタデータを拡張し、[FAISS](https://faiss.ai/index.html)を適用する `index_docs` 関数を作成します。埋め込みはローカルに保存されます。

In [ ]:
from typing import List, Union


def index_docs(embeddings_model, url: Union[str, bytes], splitter, documents: List[str], dest_embed_dir: str) -> None:
    """
    Split the documents into chunks and create embeddings for them.
    
    Args:
        embeddings_model: Model used for creating embeddings.
        url: Source url for the documents.
        splitter: Splitter used to split the documents.
        documents: List of documents whose embeddings need to be created.
        dest_embed_dir: Destination directory for embeddings.
    """
    texts = []
    metadatas = []

    for document in documents:
        chunk_texts = splitter.split_text(document.page_content)
        texts.extend(chunk_texts)
        metadatas.extend([document.metadata] * len(chunk_texts))

    if os.path.exists(dest_embed_dir):
        docsearch = FAISS.load_local(
            folder_path=dest_embed_dir, 
            embeddings=embeddings_model, 
            allow_dangerous_deserialization=True
        )
        docsearch.add_texts(texts, metadatas=metadatas)
    else:
        docsearch = FAISS.from_texts(texts, embedding=embeddings_model, metadatas=metadatas)

    docsearch.save_local(folder_path=dest_embed_dir)

#### ベクターストアからの埋め込みのロードとNVIDIA Endpointを使用したRAGの構築

次に、関数 `create_embeddings` を呼び出し、FAISSを使って[vector store](https://developer.nvidia.com/blog/accelerating-vector-search-fine-tuning-gpu-index-algorithms/)から文書を読み込む。ベクトルストアはembeddingsと呼ばれる高次元空間に関連情報を格納しています。

以下の2つのセルを実行してください。

In [ ]:
%%time
create_embeddings(embeddings_model=embeddings_model)

In [ ]:
# load Embed documents
embedding_path = "./embed/"
docsearch = FAISS.load_local(folder_path=embedding_path, embeddings=embeddings_model, allow_dangerous_deserialization=True)

### llama-3.1-swallow-8b-instructで会話型検索チェーンを作る

デプロイされたNIMは`http://0.0.0.0:8000`で稼働しているので、NIMの基本モデル`llama-3.1-swallow-8b-instruct`に基づいて[会話型検索チェーン](https://python.langchain.com/v0.1/docs/modules/chains/#conversationalretrievalchain-with-streaming-to-stdout)を作成する。

In [ ]:
llm = ChatNVIDIA(base_url="http://0.0.0.0:{}/v1".format(os.environ['CONTAINER_PORT']),
                 model="tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1", temperature=0.0, max_tokens=1000, top_p=1.0)

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

qa_prompt=QA_PROMPT

doc_chain = load_qa_chain(llm, chain_type="stuff", prompt=QA_PROMPT)

qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=docsearch.as_retriever(),
    chain_type="stuff",
    memory=memory,
    combine_docs_chain_kwargs={'prompt': qa_prompt},
)

### クエリによるテスト

In [ ]:
query = "世帯主が不意の事故により入院が必要になる場合の必要資金について、60～64歳及び65歳以上の夫婦が公的年金以 \
        外に必要とする月間生活資金と比較してください。"
result = qa({"question": query})
print(result.get("answer"))

In [ ]:
query = "製造たばこの個包装及び中間包装に求められる識別マークの表示方法の条件について説明してください。"
result = qa({"question": query})
print(result.get("answer"))

先に進む前に、dockerコンテナを停止してGPUのVRAMを解放しましょう。

In [ ]:
! docker container stop llm_nim

次のノートブックでは、LoRAのようなPEFTの機能をNIMに追加する方法を説明します。

---

## References

- https://developer.nvidia.com/blog/tips-for-building-a-rag-pipeline-with-nvidia-ai-langchain-ai-endpoints/
- https://nvidia.github.io/GenerativeAIExamples/latest/notebooks/05_RAG_for_HTML_docs_with_Langchain_NVIDIA_AI_Endpoints.html

## Licensing

Copyright © 2024 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.

<br>
<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="rag_nim_endpoints.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="rag_nim_endpoints.ipynb">1</a>
        <a >2</a>
        <a href="nim_lora_adapter.ipynb">3</a>
        <!-- <a href="challenge.ipynb">4</a> -->
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="nim_lora_adapter.ipynb">Next Notebook</a></span>
</div>

<br>
<p> <center> <a href="../../Start-NIM-RAG.ipynb">Home Page</a> </center> </p>